# Diffusion-Pipe Training Notebook
Optimized for 4x A4000 GPU Servers

### Cell 1: Token Setup
Edit your tokens below. They are saved to `/home/jovyan/.tokens.env` so you only need to set them once.

### Cell 2: GPU Selection
Edit the GPU list below. Set to `"0,1,2,3"` for all 4 GPUs, or e.g. `"0,1"` for just 2.

In [8]:
## ========== GPU Selector ==========
## Pick which GPU to use on this shared A4000 server.
## Set GPU_ID to 0, 1, 2, 3 before running any other cells.

import os

GPU_ID = "2,3" # <-- Change this to the GPU you want (0, 1, 2, 3)

os.environ['CUDA_DEVICE_ORDER'] = 'PCI_BUS_ID'
os.environ['CUDA_VISIBLE_DEVICES'] = str(GPU_ID)

print(f'\u2705 Using GPU {GPU_ID}')
!nvidia-smi -i $GPU_ID --query-gpu=index,name,memory.used,memory.total,utilization.gpu --format=csv


✅ Using GPU 2,3
index, name, memory.used [MiB], memory.total [MiB], utilization.gpu [%]
2, NVIDIA RTX A4000, 1 MiB, 16376 MiB, 0 %
3, NVIDIA RTX A4000, 4 MiB, 16376 MiB, 0 %


### Cell 3: Diffusion-Pipe Installation
Edit the repo URL if using a fork. Run once to clone and install everything.

In [4]:
import os
import subprocess

# ============================================================
#  EDIT REPO URL IF USING A FORK
# ============================================================
REPO_URL = "https://github.com/bluvoll/diffusion-pipe"

# ============================================================

def run_cmd(cmd):
    print(f"Running: {cmd}")
    subprocess.run(cmd, shell=True, check=True)

print("⏳ Starting Installation...")
os.chdir('/home/jovyan')

if not os.path.exists('diffusion-pipe'):
    run_cmd(f"git clone --recurse-submodules {REPO_URL}")
else:
    print("diffusion-pipe already cloned.")

os.chdir('/home/jovyan/diffusion-pipe')

# Install system tools
try:
    run_cmd("sudo apt-get update && sudo apt-get install -y aria2")
except:
    print("   ⚠️ Could not install aria2, wget will be used as fallback")

# Install Python dependencies
try:
    run_cmd("pip install --upgrade pip")
    run_cmd("pip install torch torchvision")
    run_cmd("pip install -r requirements.txt")
except Exception as e:
    print(f"Pip error: {e}")

# Install CUDA nvcc compiler via conda (needed by DeepSpeed)
print()
print("🔧 Installing CUDA toolkit (nvcc) via conda...")
try:
    subprocess.run("conda install -y -c nvidia cuda-nvcc --no-update-deps", shell=True, check=True)
except:
    print("   conda install failed, trying apt fallback...")
    subprocess.run("sudo apt-get update && sudo apt-get install -y nvidia-cuda-toolkit", shell=True, check=False)

# Find nvcc and set CUDA_HOME
try:
    nvcc_bin = subprocess.check_output("which nvcc", shell=True, text=True).strip()
    cuda_home = os.path.dirname(os.path.dirname(nvcc_bin))
    os.environ['CUDA_HOME'] = cuda_home
    print(f"   CUDA_HOME = {cuda_home}")
    nvcc_ver = subprocess.check_output("nvcc --version", shell=True, text=True)
    for ver_line in nvcc_ver.strip().splitlines():
        if 'release' in ver_line.lower():
            print(f"   {ver_line.strip()}")
    print("   ✅ CUDA toolkit ready!")
except Exception as e:
    print(f"   ⚠️ nvcc not found: {e}")
    os.environ['CUDA_HOME'] = '/usr/local/cuda'

print()
print("✅ Installation Complete!")
print("ℹ️ flash-attn skipped (no prebuilt wheel for Py3.13 + Torch 2.9). PyTorch SDPA used instead.")


⏳ Starting Installation...
diffusion-pipe already cloned.
Running: sudo apt-get update && sudo apt-get install -y aria2
Hit:1 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:2 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:3 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Reading package lists...
Reading package lists...
Building dependency tree...
Reading state information...
aria2 is already the newest version (1.37.0+debian-1build3).
0 upgraded, 0 newly installed, 0 to remove and 133 not upgraded.
Running: pip install --upgrade pip
  Using cached pip-26.1.2-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.1.2-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 25.2
    Uninstalling pip-25.2:
      Successfully uninstalled pip-25.2
Running: pip install torch torchvision
Running: pip install -r requirements.txt
  Using cached deepspeed-0.18.4-py3-

/opt/conda/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Retrieving notices: done


/opt/conda/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Channels:
 - nvidia
 - conda-forge
Platform: linux-64
Solving environment: done




==> WARNING: A newer version of conda exists. <==
    current version: 25.9.1
    latest version: 26.5.3

Please update conda by running

    $ conda update -n base -c conda-forge conda





## Package Plan ##

  environment location: /opt/conda

  added / updated specs:
    - cuda-nvcc


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    binutils_impl_linux-64-2.44|       h9d8b0ac_4         3.5 MB  conda-forge
    binutils_linux-64-2.44     |       h4852527_4          35 KB  conda-forge
    ca-certificates-2026.6.17  |       hbd8a1cb_0         126 KB  conda-forge
    certifi-2026.6.17          |     pyhd8ed1ab_0         131 KB  conda-forge
    cuda-cccl_linux-64-12.9.27 |                0         1.1 MB  nvidia
    cuda-crt-dev_linux-64-12.9.86|                0          84 KB  nvidia
    cuda-crt-tools-12.9.86     |                0          20 KB  nvidia
    cuda-cudart-12.9.79        |                0          17 KB  nvidia
    cuda-cudart-dev-12.9.79    |                0          17 KB  nvidia
    cuda-cudart-dev_linux-64-12.9.79|                0         374 KB  nvidia

### Cell 4: Directory Setup
Creates all necessary directories. Run once.

In [5]:
import os

directories = [
    '/home/jovyan/models/',
    '/home/jovyan/models/vae/',
    '/home/jovyan/models/text_encoder/',
    '/home/jovyan/models/transformer/',
    '/home/jovyan/datasets/',
    '/home/jovyan/outputs/',
    '/home/jovyan/configs/',
    '/tmp/diffusion-pipe-cache/'
]

for d in directories:
    os.makedirs(d, exist_ok=True)
    print(f"📁 {d}")

print("✅ Directory setup complete!")


📁 /home/jovyan/models/
📁 /home/jovyan/models/vae/
📁 /home/jovyan/models/text_encoder/
📁 /home/jovyan/models/transformer/
📁 /home/jovyan/datasets/
📁 /home/jovyan/outputs/
📁 /home/jovyan/configs/
📁 /tmp/diffusion-pipe-cache/
✅ Directory setup complete!


### Cell 5: Model / VAE / Text Encoder Download
Edit the URLs below directly. Supports HuggingFace `/resolve/main/` links and CivitAI links.
Leave a URL as `""` to skip. Uses `aria2c` for fast downloads.

In [6]:
import os
import subprocess
import shutil

HOME = '/home/jovyan'

# ============================================================
#  SECTION 1: SINGLE FILE DOWNLOADS
#  Edit URLs below — leave as "" to skip
# ============================================================

# Transformer / main model checkpoint
model_url  = 'https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/diffusion_models/anima-base-v1.0.safetensors'
model_name = ''   # leave '' to auto-detect from URL

# VAE
vae_url  = 'https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/vae/qwen_image_vae.safetensors'
vae_name = ''

# Text encoder — single safetensors file (use this OR the HF repo below, not both)
te_url  = ''
te_name = ''

# ============================================================
#  SECTION 2: FULL HF REPO DOWNLOADS
#  For models that need config.json + tokenizer files (e.g. Qwen3-0.6B)
#  Set repo_id to '' to skip
# ============================================================

# Qwen3-0.6B full HF directory (needed for qwen_path in training config)
qwen_repo_id   = 'Qwen/Qwen3-0.6B'
qwen_repo_dest = f'{HOME}/models/text_encoder/qwen3-06b'

# ============================================================

hf_token      = os.environ.get('HF_TOKEN', '')
civitai_token = os.environ.get('CIVITAI_API_TOKEN', '')
HAS_ARIA2     = shutil.which('aria2c') is not None

print('📥 Single-file downloader: ' + ('aria2c (16-connection parallel)' if HAS_ARIA2 else 'wget (fallback)'))


# ── Single file download ────────────────────────────────────────────────────

def download_file(url, dest_dir, custom_name=''):
    if not url:
        return
    os.makedirs(dest_dir, exist_ok=True)
    fname = custom_name or url.split('/')[-1].split('?')[0]
    dest_path = os.path.join(dest_dir, fname)

    if os.path.exists(dest_path):
        size_mb = os.path.getsize(dest_path) / (1024 * 1024)
        print(f'⏭️  Skipping {fname} (already exists, {size_mb:.0f} MB)')
        return

    download_url = url
    if 'huggingface.co' in url and hf_token:
        sep = '&' if '?' in url else '?'
        download_url += f'{sep}token={hf_token}'
    elif 'civitai.com' in url and civitai_token:
        sep = '&' if '?' in url else '?'
        download_url += f'{sep}token={civitai_token}'

    print(f'\n⏳ Downloading: {fname}')
    print(f'   → {dest_dir}')

    if HAS_ARIA2:
        cmd = (
            f'aria2c --console-log-level=error --summary-interval=10 '
            f'-c -x 16 -s 16 -k 1M '
            f'-d "{dest_dir}" -o "{fname}" "{download_url}"'
        )
    else:
        cmd = f'wget -c -O "{dest_path}" "{download_url}"'

    r = subprocess.run(cmd, shell=True)
    if r.returncode == 0:
        size_mb = os.path.getsize(dest_path) / (1024 * 1024)
        print(f'   ✅ Done! ({size_mb:.1f} MB)')
    else:
        print(f'   ❌ Failed (exit code {r.returncode})')


# ── Full HF repo download ───────────────────────────────────────────────────

def download_hf_repo(repo_id, dest_dir):
    if not repo_id:
        return

    os.makedirs(dest_dir, exist_ok=True)

    existing = [f for f in os.listdir(dest_dir) if not f.startswith('.')]
    if existing:
        print(f'⏭️  Skipping {repo_id} (destination already has {len(existing)} files)')
        return

    print(f'\n⏳ Downloading HF repo: {repo_id}')
    print(f'   → {dest_dir}')

    try:
        from huggingface_hub import snapshot_download
        snapshot_download(
            repo_id=repo_id,
            local_dir=dest_dir,
            token=hf_token or None,
            ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'tf_model*'],
        )
        print(f'   ✅ Done!')

    except ImportError:
        print('   huggingface_hub not found, falling back to huggingface-cli...')
        token_flag = f'--token {hf_token}' if hf_token else ''
        cmd = f'huggingface-cli download {repo_id} --local-dir "{dest_dir}" {token_flag}'
        r = subprocess.run(cmd, shell=True)
        if r.returncode == 0:
            print(f'   ✅ Done!')
        else:
            print(f'   ❌ Failed (exit code {r.returncode})')


# ── Run all downloads ───────────────────────────────────────────────────────

download_file(model_url, f'{HOME}/models/transformer/', model_name)
download_file(vae_url,   f'{HOME}/models/vae/',         vae_name)
download_file(te_url,    f'{HOME}/models/text_encoder/', te_name)

download_hf_repo(qwen_repo_id, qwen_repo_dest)

print('\n✅ All downloads finished!')

📥 Single-file downloader: aria2c (16-connection parallel)
⏭️  Skipping anima-base-v1.0.safetensors (already exists, 3988 MB)
⏭️  Skipping qwen_image_vae.safetensors (already exists, 242 MB)
⏭️  Skipping Qwen/Qwen3-0.6B (destination already has 7 files)

✅ All downloads finished!


### Cell 6: Dataset Download & Unzip
Edit the variables below. Downloads a zip from HuggingFace, extracts it, and cleans up.

In [ ]:
import os
import subprocess
from zipfile import ZipFile

# ============================================================
#  EDIT DATASET INFO BELOW
# ============================================================
dataset_repo = ""          # e.g. "username/my-dataset"
dataset_file = ""          # e.g. "dataset.zip"
output_folder = ""         # e.g. "my_lora_dataset"

# ============================================================

if not all([dataset_repo, dataset_file, output_folder]):
    print("⚠️ Fill in dataset_repo, dataset_file, and output_folder above, then re-run.")
else:
    dest_dir = f"/home/jovyan/datasets/{output_folder}"
    os.makedirs(dest_dir, exist_ok=True)

    print(f"⏳ Downloading {dataset_file} from {dataset_repo}...")
    token = os.environ.get('HF_TOKEN', '')
    cmd = f"hf download {dataset_repo} {dataset_file} --local-dir /tmp --repo-type dataset"
    if token:
        cmd += f" --token {token}"
    subprocess.run(cmd, shell=True)

    zip_path = f"/tmp/{dataset_file}"
    if os.path.exists(zip_path):
        print(f"📦 Extracting to {dest_dir}...")
        with ZipFile(zip_path, 'r') as zf:
            zf.extractall(dest_dir)
        os.remove(zip_path)
        print("✅ Extracted successfully!")
    else:
        print("❌ Download failed, zip not found.")


### Cell 8: Training
Launches DeepSpeed training. Edit GPU/config settings if needed.
RTX 4000 series requires `NCCL_P2P_DISABLE=1` and `NCCL_IB_DISABLE=1`.

In [10]:
import os, subprocess
!pip install "protobuf>=6.32.1"
# ============================================================
#  EDIT TRAINING SETTINGS IF NEEDED
# ============================================================
CONFIG_PATH = "/home/jovyan/configs/anima-phase2-LOKR.toml"
TRAIN_SCRIPT = "/home/jovyan/diffusion-pipe/train.py"

# ============================================================

# Auto-detect CUDA_HOME
if 'CUDA_HOME' not in os.environ:
    try:
        nvcc_bin = subprocess.check_output("which nvcc", shell=True, text=True).strip()
        os.environ['CUDA_HOME'] = os.path.dirname(os.path.dirname(nvcc_bin))
    except:
        os.environ['CUDA_HOME'] = '/usr/local/cuda'

cuda_home = os.environ['CUDA_HOME']


import os
import subprocess
import re
import sys
from tqdm.notebook import tqdm

# ============================================================
# BUILD COMMAND
# ============================================================
train_cmd = (
    f'CUDA_HOME={cuda_home} '
    f'PYTORCH_ALLOC_CONF=expandable_segments:True '
    f'NCCL_P2P_DISABLE="1" '
    f'NCCL_IB_DISABLE="1" '
    f'TORCH_CUDNN_SDP_ALLOW_FP16=1 '
    f'TORCH_FLASH_SDP_DISABLE=1 '
    f'TORCH_MATH_SDP_DISABLE=1 '
    f'deepspeed --include localhost:2,3 '
    f'{TRAIN_SCRIPT} '
    f'--deepspeed '
    f'--config {CONFIG_PATH} '
    # f'--resume_from_checkpoint "20260705_09-37-49"'
)

# Regex patterns
train_pattern = re.compile(r"steps:\s+(?P<step>\d+)\s+loss:\s+(?P<loss>[\d\.]+)\s+iter time.*?samples/sec:\s+(?P<speed>[\d\.]+)")
spam_filters = ["FutureWarning: _check_is_size", "torch._check_is_size(blocksize)", "guard_size_oblivious", "bitsandbytes/_ops.py"]

process = subprocess.Popen(
    train_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)

# Initialize with total=None (This makes it dynamic/indeterminate)
pbar = tqdm(total=None, desc="Training", unit="step", position=0, leave=True)

try:
    for line in iter(process.stdout.readline, ''):
        cleaned_line = line.strip()
        if not cleaned_line: continue
        
        # 1. Filter known spam
        if any(spam in cleaned_line for spam in spam_filters): continue
        
        # 2. Check for Training Metrics
        train_match = train_pattern.search(cleaned_line)
        if train_match:
            step = int(train_match.group("step"))
            loss = float(train_match.group("loss"))
            speed = float(train_match.group("speed"))
            
            # Update the progress bar to the current step count
            pbar.n = step
            pbar.last_print_n = step
            pbar.set_postfix({"Loss": f"{loss:.4f}", "Img/sec": f"{speed:.3f}", "Cur_Step": step})
            pbar.refresh()
            continue
            
        # 3. Suppress internal progress bars to stop layout breaking
        if "Training Progress:" in cleaned_line or "%|" in cleaned_line:
            continue
            
        # 4. Redirect all logs to tqdm.write
        tqdm.write(cleaned_line)

except KeyboardInterrupt:
    tqdm.write("\n🛑 Manually interrupted.")
    process.terminate()
finally:
    pbar.close()
    rc = process.poll()
    if rc == 0: tqdm.write("\n🎉 Process finished successfully!")

Training: 0step [00:00, ?step/s]

[2026-07-16 08:47:11,505] [WARNING] [runner.py:232:fetch_hostfile] Unable to find hostfile, will proceed with training with local resources only.
Detected VISIBLE_DEVICES=2,3 but ignoring it because one or several of --include/--exclude/--num_gpus/--num_nodes cl args were used. If you want to use CUDA_VISIBLE_DEVICES don't pass any of these arguments to deepspeed.
[2026-07-16 08:47:11,505] [INFO] [runner.py:630:main] cmd = /opt/conda/bin/python3.13 -u -m deepspeed.launcher.launch --world_info=eyJsb2NhbGhvc3QiOiBbMiwgM119 --master_addr=127.0.0.1 --master_port=29500 --enable_each_rank_log=None --log_level=info /home/jovyan/diffusion-pipe/train.py --deepspeed --config /home/jovyan/configs/anima-phase2-LOKR.toml --resume_from_checkpoint 20260705_09-37-49
[2026-07-16 08:47:17,131] [INFO] [launch.py:155:main] 0 NCCL_P2P_DISABLE=1
[2026-07-16 08:47:17,131] [INFO] [launch.py:155:main] 0 NCCL_IB_DISABLE=1
[2026-07-16 08:47:17,131] [INFO] [launch.py:162:main] WORLD INFO DICT: {'localhost': [2, 3

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[2026-07-17 02:49:01,263] [INFO] [logging.py:123:log_dist] [Rank 0] step=13147, skipped=0, lr=[3e-06, 3e-06, 3e-06], mom=[[0.9, 0.999], [0.9, 0.999], [0.9, 0.999]]
[2026-07-17 02:49:29,071] [INFO] [logging.py:123:log_dist] [Rank 0] step=13148, skipped=0, lr=[3e-06, 3e-06, 3e-06], mom=[[0.9, 0.999], [0.9, 0.999], [0.9, 0.999]]
[2026-07-17 02:49:56,603] [INFO] [logging.py:123:log_dist] [Rank 0] step=13149, skipped=0, lr=[3e-06, 3e-06, 3e-06], mom=[[0.9, 0.999], [0.9, 0.999], [0.9, 0.999]]
[2026-07-17 02:50:12,986] [INFO] [logging.py:123:log_dist] [Rank 0] step=13150, skipped=0, lr=[3e-06, 3e-06, 3e-06], mom=[[0.9, 0.999], [0.9, 0.999], [0.9, 0.999]]
[2026-07-17 02:50:40,861] [INFO] [logging.py:123:log_dist] [Rank 0] step=13151, skipped=0, lr=[3e-06, 3e-06, 3e-06], mom=[[0.9, 0.999], [0.9, 0.999], [0.9, 0.999]]
[2026-07-17 02:50:57,362] [INFO] [logging.py:123:log_dist] [Rank 0] step=13152, skipped=0, lr=[3e-06, 3e-06, 3e-06], mom=[[0.9, 0.999], [0.9, 0.999], [0.9, 0.999]]
[2026-07-17 02:5

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[2026-07-18 04:37:41,692] [INFO] [logging.py:123:log_dist] [Rank 0] step=17403, skipped=0, lr=[3e-06, 3e-06, 3e-06], mom=[[0.9, 0.999], [0.9, 0.999], [0.9, 0.999]]
[2026-07-18 04:37:57,619] [INFO] [logging.py:123:log_dist] [Rank 0] step=17404, skipped=0, lr=[3e-06, 3e-06, 3e-06], mom=[[0.9, 0.999], [0.9, 0.999], [0.9, 0.999]]
[2026-07-18 04:38:13,627] [INFO] [logging.py:123:log_dist] [Rank 0] step=17405, skipped=0, lr=[3e-06, 3e-06, 3e-06], mom=[[0.9, 0.999], [0.9, 0.999], [0.9, 0.999]]
[2026-07-18 04:38:40,095] [INFO] [logging.py:123:log_dist] [Rank 0] step=17406, skipped=0, lr=[3e-06, 3e-06, 3e-06], mom=[[0.9, 0.999], [0.9, 0.999], [0.9, 0.999]]
[2026-07-18 04:39:06,168] [INFO] [logging.py:123:log_dist] [Rank 0] step=17407, skipped=0, lr=[3e-06, 3e-06, 3e-06], mom=[[0.9, 0.999], [0.9, 0.999], [0.9, 0.999]]
[2026-07-18 04:39:22,096] [INFO] [logging.py:123:log_dist] [Rank 0] step=17408, skipped=0, lr=[3e-06, 3e-06, 3e-06], mom=[[0.9, 0.999], [0.9, 0.999], [0.9, 0.999]]
[2026-07-18 04:3

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



### Cell 9: Upload Outputs
Scans for safetensors files and uploads them to HuggingFace.

In [11]:
import os
import subprocess

# =======================================================================
# EDIT YOUR HF REPO CONFIG & FILE PATH BELOW
# =======================================================================
HF_UPLOAD_REPO = "RicemanT/Anima-Telescopa"  # e.g. "username/my-Lora-model"
PRIVATE_REPO = True   # Set to True for PRIVATE, False for PUBLIC

# Paste the absolute path to your specific file here
TARGET_FILE = "/home/jovyan/ComfyUI/models/loras/TelescopaLOKRV0.5-Epoch10.safetensors"
# =======================================================================

# Check if the file exists locally
if not os.path.exists(TARGET_FILE):
    print(f"❌ Error: File not found at '{TARGET_FILE}'")
elif not HF_UPLOAD_REPO:
    print("⚠️ Set HF_UPLOAD_REPO above to enable uploading.")
else:
    # Calculate file size
    size = os.path.getsize(TARGET_FILE) / (1024 * 1024)
    file_name = os.path.basename(TARGET_FILE)
    
    print(f"📂 Target file found: {file_name} ({size:.2f} MB)")
    print(f"⏳ Uploading to {HF_UPLOAD_REPO}...")
    
    # Determine privacy flag
    privacy_flag = "--private" if PRIVATE_REPO else "--no-private"
    
    # Construct and run the command
    cmd = f"hf upload {HF_UPLOAD_REPO} {TARGET_FILE} {file_name} {privacy_flag}"
    subprocess.run(cmd, shell=True)
    
    print("✅ Upload complete!")


📂 Target file found: TelescopaLOKRV0.5-Epoch10.safetensors (840.07 MB)
⏳ Uploading to RicemanT/Anima-Telescopa...


Hint: A new version of huggingface_hub (1.24.0) is available! You are using version 1.23.0.
To update, run: hf update
Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...RV0.5-Epoch10.safetensors:   0%|          | 66.7kB /  881MB            

Processing Files (0 / 1)      :   0%|          | 66.7kB /  881MB,   ???B/s  

  ...RV0.5-Epoch10.safetensors:   0%|          | 66.7kB /  881MB            

  ...RV0.5-Epoch10.safetensors:   0%|          | 66.7kB /  881MB            

  ...RV0.5-Epoch10.safetensors:   0%|          | 66.7kB /  881MB            

  ...RV0.5-Epoch10.safetensors:   0%|          | 66.7kB /  881MB            

  ...RV0.5-Epoch10.safetensors:   0%|          | 66.7kB /  881MB            

  ...RV0.5-Epoch10.safetensors:   0%|          | 66.7kB /  881MB            

  ...RV0.5-Epoch10.safetensors:   0%|          | 66.7kB /  881MB            

Processing Files (0 / 1)      :  

✓ Uploaded
  url: https://huggingface.co/RicemanT/Anima-Telescopa/commit/242fe6a54c1dbaf15d92e401372c0610db37d48e
✅ Upload complete!


In [3]:
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 --upgrade

Looking in indexes: https://download.pytorch.org/whl/cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 18.2 MB/s  0:00:00
  Attempting uninstall: torchaudio
    Found existing installation: torchaudio 2.11.0
    Uninstalling torchaudio-2.11.0:
      Successfully uninstalled torchaudio-2.11.0
Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
import torchaudio
print(torch.cuda.is_available())       # True
print(torch.version.cuda)              # 12.8
print(torchaudio.version.__version__)  # should print without error

True
12.8
2.11.0+cu128


In [5]:
# Flatten Illustration - moves all images/txts up, keeps filenames
cd /home/jovyan/Anime-Background-Finetuning/Illustration
for artist_dir in */; do
    mv "$artist_dir"* . 2>/dev/null
    rmdir "$artist_dir" 2>/dev/null
done

# Do the same for Screencap
cd /home/jovyan/Anime-Background-Finetuning/Screencap
for artist_dir in */; do
    mv "$artist_dir"* . 2>/dev/null
    rmdir "$artist_dir" 2>/dev/null
done

SyntaxError: invalid syntax (1583331335.py, line 3)

In [3]:
!find / -iname "*merged_model*" 2>/dev/null


/home/jovyan/Notebook-Scripts/merged_models
